# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library, referencing all dataset entities by their Croissant schema `@id`.

### Dataset Source
The dataset source is described using a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We will:  
- List available record sets and their IDs.
- For each record set, list its fields and corresponding field IDs.  
- For each field, show the underlying columns and their IDs if available.

In [ ]:
# Get all record sets' @ids and names
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in the dataset. The dataset may be entirely tabular or expose as a single record set.")
else:
    print(f"Found {len(record_sets)} record sets:")
    for rs in record_sets:
        print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
        if hasattr(rs, 'fields'):
            for fld in rs.fields:
                col_ids = []
                if hasattr(fld, 'columns'):
                    col_ids = [col.id for col in fld.columns]
                print(f"  - Field name: {fld.name}, @id: {fld.id}, columns: {col_ids}")
        print()

# If no named record sets, try getting the default
if not record_sets:
    print("Fallback: attempting to read records anyway via dataset.records()...")
    try:
        first = next(dataset.records())
        pprint(first)
    except Exception as e:
        print("No records found or dataset does not expose row-wise iteration.")

## 3. Data Extraction
Load data from the record set(s) into DataFrames. All references are by Croissant `@id`.

If there is a main record set, or only one, we use its `@id`. Otherwise, you can adapt this cell to work with multiple record sets.

In [ ]:
# If record sets exist, extract their @id
record_set_ids = [rs.id for rs in record_sets] if record_sets else []

# Fallback: dataset may have a single tabular part even if record_sets is empty
if not record_set_ids:
    # Try loading records without specifying record_set
    records = list(dataset.records())
    if records:
        df = pd.DataFrame(records)
        print("Loaded main dataset Table (no named record_set available):")
        print(df.columns.tolist())
        display(df.head())
        # Set artificial record set id for later reference
        main_rs_id = 'default_table'
        dataframes = {main_rs_id: df}
    else:
        print("No data records available.")
else:
    dataframes = {}
    for rs_id in record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from record set {rs_id}")
        print(f"Columns: {dataframes[rs_id].columns.tolist()}")
    # Use first record set for illustration below
    main_rs_id = record_set_ids[0]
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's explore and process the tabular data.

We'll:
- Choose a numeric field (referenced by its `@id`).
- Filter records based on a numeric threshold.
- Normalize the selected numeric field among filtered records.
- Optionally group by another field (using its `@id`).

In [ ]:
# Choose the relevant DataFrame
df = dataframes[main_rs_id]

# List DataFrame columns for user reference
print("Available columns (corresponding to field @id's):")
pprint(list(df.columns))

# Suppose field @id 'Age' is present, as it's common in clinical tabular data.
# (Replace with exact @id if available. For Croissant, field or column ids are explicit.)
numeric_field_id = None
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
        break
if not numeric_field_id:
    # Fallback: try the first numeric-looking column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if numeric_field_id:
    print(f"Using numeric field (by @id): {numeric_field_id}")
    threshold = 50

    # Filter records
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - mean) / std
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by another field, e.g., 'Sex' or similar
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and ('sex' in col.lower() or 'group' in col.lower() or pd.api.types.is_categorical_dtype(df[col]) or df[col].dtype==object):
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped by {group_field_id} (by @id), mean {numeric_field_id}:")
        print(grouped_df)
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if available, the grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load dataset metadata and records using `mlcroissant` from a Croissant schema URL.
- Reference record sets and fields by their `@id`s for robust, schema-driven data handling.
- Extract, process, and visualize tabular data.

This approach supports transparent and reproducible workflows, especially important for clinical datasets like FAIR².